In [ ]:
# ============================================================
# FINAL COMPARISON — STEP 1: GPU AND DATA CHECK
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import glob
import torch

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

JOE_ARCHIVES = sorted(
    glob.glob(
        os.path.join(BASE_DIR, "joe*.gz")
    )
)

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

PREVIOUS_SOURCES = sorted(
    glob.glob(
        os.path.join(
            BASE_DIR,
            "selected_hybrid_data",
            "joe_source_*.wav"
        )
    )
)

print("=" * 70)
print("FINAL HYBRID COMPARISON — PRE-CHECK")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nJoe archives:", len(JOE_ARCHIVES))

for path in JOE_ARCHIVES:
    print(" -", path)

print("\nKathleen reference exists:", os.path.isfile(TARGET_PATH))

print("\nPrevious tuning samples:", len(PREVIOUS_SOURCES))

for path in PREVIOUS_SOURCES:
    print(" -", os.path.basename(path))

assert torch.cuda.is_available(), (
    "❌ اختاري T4 GPU"
)

assert JOE_ARCHIVES, (
    "❌ Joe archive was not found on Drive"
)

assert os.path.isfile(TARGET_PATH), (
    "❌ Kathleen reference was not found"
)

assert len(PREVIOUS_SOURCES) == 3, (
    "❌ The three tuning samples were not found"
)

print("\n✅ GPU and all required files are ready")

Mounted at /content/drive
FINAL HYBRID COMPARISON — PRE-CHECK
CUDA available: False

Joe archives: 1
 - /content/drive/MyDrive/IJEEE_VC_data/joe.tar.gz

Kathleen reference exists: True

Previous tuning samples: 3
 - joe_source_01.wav
 - joe_source_02.wav
 - joe_source_03.wav


AssertionError: ❌ اختاري T4 GPU

In [ ]:
# ============================================================
# FINAL COMPARISON — STEP 2: SELECT 10 UNSEEN JOE SAMPLES
# ============================================================

import os
import glob
import tarfile
import random
import shutil
import subprocess

from pathlib import Path

import pandas as pd

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

JOE_ARCHIVE = os.path.join(
    BASE_DIR,
    "joe.tar.gz"
)

LOCAL_EXTRACT_DIR = "/content/final_comparison_dataset"
LOCAL_JOE_DIR = os.path.join(
    LOCAL_EXTRACT_DIR,
    "joe"
)

LOCAL_HOLDOUT_DIR = "/content/final_holdout_data"

DRIVE_HOLDOUT_DIR = os.path.join(
    BASE_DIR,
    "final_holdout_data"
)

METADATA_PATH = os.path.join(
    DRIVE_HOLDOUT_DIR,
    "holdout_metadata.csv"
)

os.makedirs(LOCAL_JOE_DIR, exist_ok=True)
os.makedirs(LOCAL_HOLDOUT_DIR, exist_ok=True)
os.makedirs(DRIVE_HOLDOUT_DIR, exist_ok=True)

assert os.path.isfile(JOE_ARCHIVE)
assert shutil.which("ffprobe") is not None
assert shutil.which("ffmpeg") is not None

# ------------------------------------------------------------
# Extract Joe archive locally
# ------------------------------------------------------------

existing_audio = glob.glob(
    os.path.join(
        LOCAL_JOE_DIR,
        "**",
        "*.webm"
    ),
    recursive=True
)

if not existing_audio:

    print("Extracting Joe dataset...")

    with tarfile.open(
        JOE_ARCHIVE,
        mode="r:gz"
    ) as archive:

        archive.extractall(
            LOCAL_JOE_DIR,
            filter="data"
        )

    print("✅ Joe extraction completed")

# ------------------------------------------------------------
# Find audio files
# ------------------------------------------------------------

AUDIO_EXTENSIONS = [
    "*.webm",
    "*.wav",
    "*.flac",
    "*.mp3",
    "*.ogg",
    "*.opus",
    "*.m4a"
]

joe_audio_files = []

for extension in AUDIO_EXTENSIONS:

    joe_audio_files.extend(
        glob.glob(
            os.path.join(
                LOCAL_JOE_DIR,
                "**",
                extension
            ),
            recursive=True
        )
    )

joe_audio_files = sorted(set(joe_audio_files))

print("Joe audio files found:", len(joe_audio_files))

assert len(joe_audio_files) >= 1000, (
    f"❌ Expected approximately 1150 files, "
    f"found {len(joe_audio_files)}"
)

# ------------------------------------------------------------
# Exclude the three tuning samples
# ------------------------------------------------------------

TUNING_SAMPLE_IDS = {
    "0000000160",
    "0000000229",
    "0000000068"
}

candidate_files = [
    path
    for path in joe_audio_files
    if Path(path).stem not in TUNING_SAMPLE_IDS
]

# ثابت لضمان اختيار نفس العينات إذا أعدنا التجربة
selection_rng = random.Random(20260822)
selection_rng.shuffle(candidate_files)

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def get_audio_duration(audio_path):

    result = subprocess.run(
        [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            audio_path
        ],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        return None

    try:
        return float(result.stdout.strip())
    except (TypeError, ValueError):
        return None


def read_transcript(audio_path):

    transcript_path = str(
        Path(audio_path).with_suffix(".txt")
    )

    if not os.path.isfile(transcript_path):
        return None

    with open(
        transcript_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as transcript_file:

        transcript = transcript_file.read().strip()

    transcript = " ".join(
        transcript.split()
    )

    return transcript if transcript else None


def convert_to_fixed_wav(
    source_path,
    output_path
):

    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-v", "error",
            "-i", source_path,
            "-ac", "1",
            "-ar", "22050",
            "-c:a", "pcm_s16le",
            output_path
        ],
        check=True
    )

# ------------------------------------------------------------
# Select exactly 10 valid unseen samples
# ------------------------------------------------------------

selected_records = []

for source_path in candidate_files:

    if len(selected_records) == 10:
        break

    duration = get_audio_duration(source_path)

    if duration is None:
        continue

    if not (4.0 < duration < 7.5):
        continue

    transcript = read_transcript(source_path)

    if transcript is None:
        continue

    sample_number = len(selected_records) + 1

    local_output_path = os.path.join(
        LOCAL_HOLDOUT_DIR,
        f"joe_holdout_{sample_number:02d}.wav"
    )

    drive_output_path = os.path.join(
        DRIVE_HOLDOUT_DIR,
        f"joe_holdout_{sample_number:02d}.wav"
    )

    convert_to_fixed_wav(
        source_path,
        local_output_path
    )

    shutil.copy2(
        local_output_path,
        drive_output_path
    )

    selected_records.append({
        "sample": sample_number,
        "original_id": Path(source_path).stem,
        "duration_seconds": duration,
        "transcript": transcript,
        "prepared_wav": drive_output_path,
        "original_archive_path": os.path.relpath(
            source_path,
            LOCAL_JOE_DIR
        )
    })

assert len(selected_records) == 10, (
    f"❌ Only {len(selected_records)} valid samples were found"
)

holdout_df = pd.DataFrame(
    selected_records
)

holdout_df.to_csv(
    METADATA_PATH,
    index=False
)

print("\n" + "=" * 90)
print("FIXED UNSEEN HOLDOUT SAMPLES")
print("=" * 90)

print(
    holdout_df[
        [
            "sample",
            "original_id",
            "duration_seconds",
            "transcript"
        ]
    ].round(
        {"duration_seconds": 3}
    ).to_string(index=False)
)

print("\n✅ Holdout samples selected:", len(holdout_df))
print("✅ Saved in:", DRIVE_HOLDOUT_DIR)
print("✅ Metadata:", METADATA_PATH)
print("✅ Previous three tuning samples were excluded")

Extracting Joe dataset...
✅ Joe extraction completed
Joe audio files found: 1150

FIXED UNSEEN HOLDOUT SAMPLES
 sample original_id  duration_seconds                                                                                    transcript
      1  4000000077              4.98                                We expect that an agent will be available to help you shortly.
      2  3000000204              4.46                                That's good, otherwise we would have to pay for the bike rack.
      3  0000000107              6.32       It is perfect for families since the kids can safely run around and play in the garden.
      4  4000000151              4.24                                    I now need to take your case to the senior representative.
      5  0300000043              4.44                                                       He earned a master's degree from N E U.
      6  0000000002              6.86 Baking is very good for focusing the mind because it often 

In [ ]:
# ============================================================
# FINAL COMPARISON — STEP 3: INSTALL AND CHECK OPENVOICE
# ============================================================

import os
import sys
import subprocess

OPENVOICE_REPO = "/content/OpenVoice"

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

CONVERTER_DIR = os.path.join(
    BASE_DIR,
    "models",
    "OpenVoiceV2",
    "converter"
)

CONFIG_PATH = os.path.join(
    CONVERTER_DIR,
    "config.json"
)

CHECKPOINT_PATH = os.path.join(
    CONVERTER_DIR,
    "checkpoint.pth"
)

# ------------------------------------------------------------
# Clone official OpenVoice repository
# ------------------------------------------------------------

if not os.path.isdir(OPENVOICE_REPO):

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/myshell-ai/OpenVoice.git",
            OPENVOICE_REPO
        ],
        check=True
    )

# ------------------------------------------------------------
# Install only the dependencies required by OpenVoice
# Avoid old incompatible NumPy/Torch requirements
# ------------------------------------------------------------

openvoice_packages = [
    "wavmark==0.0.3",
    "eng_to_ipa==0.0.2",
    "inflect==7.0.0",
    "unidecode==1.3.7",
    "pypinyin==0.50.0",
    "cn2an==0.5.22",
    "jieba==0.42.1",
    "langid==1.1.6"
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q"
    ] + openvoice_packages,
    check=True
)

# Add repository directly to Python path
if OPENVOICE_REPO not in sys.path:
    sys.path.insert(0, OPENVOICE_REPO)

# ------------------------------------------------------------
# Verify source and checkpoint
# ------------------------------------------------------------

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=OPENVOICE_REPO,
    text=True
).strip()

print("=" * 70)
print("OPENVOICE INSTALLATION CHECK")
print("=" * 70)

print("Repository:", OPENVOICE_REPO)
print("Commit:", commit)
print("OpenVoice source exists:", os.path.isdir(OPENVOICE_REPO))
print("Config exists:", os.path.isfile(CONFIG_PATH))
print("Checkpoint exists:", os.path.isfile(CHECKPOINT_PATH))

assert os.path.isdir(OPENVOICE_REPO), (
    "❌ OpenVoice source is missing"
)

assert os.path.isfile(CONFIG_PATH), (
    "❌ OpenVoice config is missing"
)

assert os.path.isfile(CHECKPOINT_PATH), (
    "❌ OpenVoice checkpoint is missing"
)

from openvoice.api import ToneColorConverter

print("\n✅ OpenVoice import successful")
print("Next: load converter and extract fixed embeddings")

OPENVOICE INSTALLATION CHECK
Repository: /content/OpenVoice
Commit: 74a1d147b17a8c3092dd5430504bd83ef6c7eb23
OpenVoice source exists: True
Config exists: True
Checkpoint exists: True


/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")



✅ OpenVoice import successful
Next: load converter and extract fixed embeddings


In [ ]:
# ============================================================
# FINAL COMPARISON — STEP 4:
# LOAD OPENVOICE AND EXTRACT FIXED EMBEDDINGS
# ============================================================

import os
import glob
import time
import torch

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

HOLDOUT_DIR = os.path.join(
    BASE_DIR,
    "final_holdout_data"
)

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

EMBEDDING_DIR = os.path.join(
    BASE_DIR,
    "final_hybrid_comparison",
    "openvoice_embeddings"
)

CONVERTER_DIR = os.path.join(
    BASE_DIR,
    "models",
    "OpenVoiceV2",
    "converter"
)

CONFIG_PATH = os.path.join(
    CONVERTER_DIR,
    "config.json"
)

CHECKPOINT_PATH = os.path.join(
    CONVERTER_DIR,
    "checkpoint.pth"
)

HOLDOUT_SOURCES = sorted(
    glob.glob(
        os.path.join(
            HOLDOUT_DIR,
            "joe_holdout_*.wav"
        )
    )
)

os.makedirs(EMBEDDING_DIR, exist_ok=True)

assert torch.cuda.is_available()
assert len(HOLDOUT_SOURCES) == 10
assert os.path.isfile(TARGET_PATH)
assert os.path.isfile(CONFIG_PATH)
assert os.path.isfile(CHECKPOINT_PATH)

DEVICE = "cuda:0"

# ------------------------------------------------------------
# Load converter
# ------------------------------------------------------------

print("Loading OpenVoice V2 converter...")

tone_color_converter = ToneColorConverter(
    CONFIG_PATH,
    device=DEVICE
)

tone_color_converter.load_ckpt(
    CHECKPOINT_PATH
)

print("\n✅ OpenVoice V2 converter loaded")
print("Device:", DEVICE)
print("Version:", tone_color_converter.version)

# ------------------------------------------------------------
# Extract Kathleen target embedding once
# ------------------------------------------------------------

print("\nExtracting fixed Kathleen embedding...")

torch.cuda.synchronize()
target_start = time.perf_counter()

target_se = tone_color_converter.extract_se(
    TARGET_PATH
)

torch.cuda.synchronize()
target_embedding_time = (
    time.perf_counter() - target_start
)

TARGET_EMBEDDING_PATH = os.path.join(
    EMBEDDING_DIR,
    "kathleen_target_se.pth"
)

torch.save(
    target_se.detach().cpu(),
    TARGET_EMBEDDING_PATH
)

print(
    "Kathleen embedding time:",
    round(target_embedding_time, 4),
    "seconds"
)

# ------------------------------------------------------------
# Extract embeddings for the 10 Joe holdout samples
# ------------------------------------------------------------

print("\nExtracting fixed Joe holdout embeddings...")

source_embeddings = {}
embedding_records = []

for sample_number, source_path in enumerate(
    HOLDOUT_SOURCES,
    start=1
):

    torch.cuda.synchronize()
    embedding_start = time.perf_counter()

    source_se = tone_color_converter.extract_se(
        source_path
    )

    torch.cuda.synchronize()
    embedding_time = (
        time.perf_counter() - embedding_start
    )

    source_embeddings[sample_number] = source_se

    embedding_path = os.path.join(
        EMBEDDING_DIR,
        f"joe_holdout_{sample_number:02d}_se.pth"
    )

    torch.save(
        source_se.detach().cpu(),
        embedding_path
    )

    embedding_records.append({
        "sample": sample_number,
        "embedding_seconds": embedding_time,
        "embedding_path": embedding_path
    })

    print(
        f"Sample {sample_number:02d}: "
        f"{embedding_time:.4f} seconds"
    )

print("\n" + "=" * 70)
print("✅ OPENVOICE EMBEDDINGS READY")
print("=" * 70)

print("Target embeddings:", 1)
print("Source embeddings:", len(source_embeddings))
print("Saved in:", EMBEDDING_DIR)
print("GPU:", torch.cuda.get_device_name(0))

Loading OpenVoice V2 converter...


/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


step59000_snr39.99_pesq4.35_BERP_none0.3(…): reconstructing file:   0%|          |  0.00B / 10.0MB            

step59000_snr39.99_pesq4.35_BERP_none0.3(…): downloading bytes:           |  0.00B            

Loaded checkpoint '/content/drive/MyDrive/IJEEE_VC_data/models/OpenVoiceV2/converter/checkpoint.pth'
missing/unexpected keys: [] []

✅ OpenVoice V2 converter loaded
Device: cuda:0
Version: v2

Extracting fixed Kathleen embedding...


/usr/local/lib/python3.13/dist-packages/torch/functional.py:681: UserWarning: stft with return_complex=False is deprecated. In a future pytorch release, stft will return complex tensors for all inputs, and return_complex=False will raise an error.
Note: you can still call torch.view_as_real on the complex output to recover the old return format. (Triggered internally at /pytorch/aten/src/ATen/native/SpectralOps.cpp:874.)
  return _VF.stft(  # type: ignore[attr-defined]


Kathleen embedding time: 15.3892 seconds

Extracting fixed Joe holdout embeddings...
Sample 01: 0.0383 seconds
Sample 02: 0.0161 seconds
Sample 03: 0.0168 seconds
Sample 04: 0.0134 seconds
Sample 05: 0.0095 seconds
Sample 06: 0.0153 seconds
Sample 07: 0.0106 seconds
Sample 08: 0.0171 seconds
Sample 09: 0.0113 seconds
Sample 10: 0.0154 seconds

✅ OPENVOICE EMBEDDINGS READY
Target embeddings: 1
Source embeddings: 10
Saved in: /content/drive/MyDrive/IJEEE_VC_data/final_hybrid_comparison/openvoice_embeddings
GPU: Tesla T4


In [ ]:
# ============================================================
# FINAL COMPARISON — STEP 5: OPENVOICE WARM-UP
# ============================================================

import os
import random
import numpy as np
import torch

WARMUP_DIR = "/content/openvoice_final_warmup"
os.makedirs(WARMUP_DIR, exist_ok=True)

WARMUP_OUTPUT = os.path.join(
    WARMUP_DIR,
    "openvoice_warmup.wav"
)

print("Running OpenVoice GPU warm-up...\n")

for warmup_run in range(1, 3):

    random.seed(2026)
    np.random.seed(2026)
    torch.manual_seed(2026)
    torch.cuda.manual_seed_all(2026)

    torch.cuda.synchronize()

    tone_color_converter.convert(
        audio_src_path=HOLDOUT_SOURCES[0],
        src_se=source_embeddings[1],
        tgt_se=target_se,
        output_path=WARMUP_OUTPUT,
        tau=0.5,
        message="@MyShell"
    )

    torch.cuda.synchronize()

    print(
        f"✅ Warm-up run {warmup_run}/2 completed"
    )

print("\n" + "=" * 70)
print("✅ OPENVOICE WARM-UP COMPLETED")
print("=" * 70)

print("Warm-up tau: 0.5")
print("GPU:", torch.cuda.get_device_name(0))

Running OpenVoice GPU warm-up...

✅ Warm-up run 1/2 completed
✅ Warm-up run 2/2 completed

✅ OPENVOICE WARM-UP COMPLETED
Warm-up tau: 0.5
GPU: Tesla T4


In [ ]:
# ============================================================
# FINAL COMPARISON — STEP 6:
# RUN OPENVOICE tau=0.3 AND tau=0.5
# ============================================================

import os
import time
import shutil
import random

import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# Paths and fixed configuration
# ------------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

HOLDOUT_DIR = os.path.join(
    BASE_DIR,
    "final_holdout_data"
)

METADATA_PATH = os.path.join(
    HOLDOUT_DIR,
    "holdout_metadata.csv"
)

COMPARISON_DIR = os.path.join(
    BASE_DIR,
    "final_hybrid_comparison"
)

OPENVOICE_OUTPUT_DIR = os.path.join(
    COMPARISON_DIR,
    "openvoice_outputs"
)

LOCAL_WORK_DIR = "/content/openvoice_final_work"

os.makedirs(OPENVOICE_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

TAU_VALUES = [0.3, 0.5]
TIMING_REPEATS = 3

SAMPLE_SEEDS = {
    sample: 202700 + sample
    for sample in range(1, 11)
}

holdout_df = pd.read_csv(
    METADATA_PATH
)

duration_map = {
    int(row["sample"]): float(row["duration_seconds"])
    for _, row in holdout_df.iterrows()
}

source_path_map = {
    int(row["sample"]): row["prepared_wav"]
    for _, row in holdout_df.iterrows()
}

assert len(holdout_df) == 10
assert len(source_embeddings) == 10
assert torch.cuda.is_available()

# ------------------------------------------------------------
# Result paths
# ------------------------------------------------------------

RAW_METRICS_PATH = os.path.join(
    COMPARISON_DIR,
    "openvoice_final_timing_repetitions.csv"
)

SAMPLE_METRICS_PATH = os.path.join(
    COMPARISON_DIR,
    "openvoice_final_sample_metrics.csv"
)

SUMMARY_PATH = os.path.join(
    COMPARISON_DIR,
    "openvoice_final_summary.csv"
)

raw_records = []
sample_records = []

print("=" * 75)
print("FINAL OPENVOICE HOLDOUT EXPERIMENT")
print("=" * 75)

print("Tau values:", TAU_VALUES)
print("Holdout samples:", len(holdout_df))
print("Timing repetitions:", TIMING_REPEATS)
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Run conversions
# ------------------------------------------------------------

for tau in TAU_VALUES:

    tau_drive_dir = os.path.join(
        OPENVOICE_OUTPUT_DIR,
        f"tau_{tau:.1f}"
    )

    tau_local_dir = os.path.join(
        LOCAL_WORK_DIR,
        f"tau_{tau:.1f}"
    )

    os.makedirs(tau_drive_dir, exist_ok=True)
    os.makedirs(tau_local_dir, exist_ok=True)

    print("\n" + "=" * 75)
    print(f"Running OpenVoice tau={tau:.1f}")
    print("=" * 75)

    for sample_number in range(1, 11):

        source_path = source_path_map[sample_number]
        duration = duration_map[sample_number]
        fixed_seed = SAMPLE_SEEDS[sample_number]

        local_output_path = os.path.join(
            tau_local_dir,
            f"openvoice_sample_{sample_number:02d}_tau_{tau:.1f}.wav"
        )

        drive_output_path = os.path.join(
            tau_drive_dir,
            f"openvoice_sample_{sample_number:02d}_tau_{tau:.1f}.wav"
        )

        repetition_latencies = []

        for repetition in range(1, TIMING_REPEATS + 1):

            # Same seed for the same sample across tau values
            random.seed(fixed_seed)
            np.random.seed(fixed_seed)
            torch.manual_seed(fixed_seed)
            torch.cuda.manual_seed_all(fixed_seed)

            torch.cuda.synchronize()
            start_time = time.perf_counter()

            tone_color_converter.convert(
                audio_src_path=source_path,
                src_se=source_embeddings[sample_number],
                tgt_se=target_se,
                output_path=local_output_path,
                tau=tau,
                message="@MyShell"
            )

            torch.cuda.synchronize()
            latency = time.perf_counter() - start_time

            rtf = latency / duration

            repetition_latencies.append(latency)

            raw_records.append({
                "tau": tau,
                "sample": sample_number,
                "repetition": repetition,
                "fixed_seed": fixed_seed,
                "duration_seconds": duration,
                "openvoice_latency_seconds": latency,
                "openvoice_rtf": rtf
            })

            # Progressive saving
            pd.DataFrame(raw_records).to_csv(
                RAW_METRICS_PATH,
                index=False
            )

            print(
                f"Tau={tau:.1f} | "
                f"Sample={sample_number:02d} | "
                f"Repeat={repetition}/{TIMING_REPEATS} | "
                f"Latency={latency:.4f}s | "
                f"RTF={rtf:.4f}"
            )

        mean_latency = float(
            np.mean(repetition_latencies)
        )

        std_latency = float(
            np.std(
                repetition_latencies,
                ddof=1
            )
        )

        mean_rtf = mean_latency / duration

        # Save the final deterministic output to Drive
        shutil.copy2(
            local_output_path,
            drive_output_path
        )

        sample_records.append({
            "tau": tau,
            "sample": sample_number,
            "fixed_seed": fixed_seed,
            "duration_seconds": duration,
            "mean_openvoice_latency_seconds": mean_latency,
            "std_openvoice_latency_seconds": std_latency,
            "mean_openvoice_rtf": mean_rtf,
            "output_path": drive_output_path
        })

        pd.DataFrame(sample_records).to_csv(
            SAMPLE_METRICS_PATH,
            index=False
        )

        print(
            f"✅ Sample {sample_number:02d} completed | "
            f"Mean latency={mean_latency:.4f}s | "
            f"RTF={mean_rtf:.4f}\n"
        )

# ------------------------------------------------------------
# Summary across the 10 samples
# ------------------------------------------------------------

sample_metrics_df = pd.DataFrame(
    sample_records
)

summary_df = (
    sample_metrics_df
    .groupby("tau", as_index=False)
    .agg(
        mean_latency=(
            "mean_openvoice_latency_seconds",
            "mean"
        ),
        std_latency=(
            "mean_openvoice_latency_seconds",
            "std"
        ),
        mean_rtf=(
            "mean_openvoice_rtf",
            "mean"
        ),
        std_rtf=(
            "mean_openvoice_rtf",
            "std"
        )
    )
)

summary_df.to_csv(
    SUMMARY_PATH,
    index=False
)

print("\n" + "=" * 75)
print("FINAL OPENVOICE HOLDOUT SUMMARY")
print("=" * 75)

print(
    summary_df.round(4).to_string(index=False)
)

print("\n✅ Outputs:", OPENVOICE_OUTPUT_DIR)
print("✅ Raw timing metrics:", RAW_METRICS_PATH)
print("✅ Sample metrics:", SAMPLE_METRICS_PATH)
print("✅ Summary:", SUMMARY_PATH)

FINAL OPENVOICE HOLDOUT EXPERIMENT
Tau values: [0.3, 0.5]
Holdout samples: 10
Timing repetitions: 3
GPU: Tesla T4

Running OpenVoice tau=0.3
Tau=0.3 | Sample=01 | Repeat=1/3 | Latency=0.3153s | RTF=0.0633
Tau=0.3 | Sample=01 | Repeat=2/3 | Latency=0.2478s | RTF=0.0498
Tau=0.3 | Sample=01 | Repeat=3/3 | Latency=0.2460s | RTF=0.0494
✅ Sample 01 completed | Mean latency=0.2697s | RTF=0.0542

Tau=0.3 | Sample=02 | Repeat=1/3 | Latency=0.2874s | RTF=0.0644
Tau=0.3 | Sample=02 | Repeat=2/3 | Latency=0.2376s | RTF=0.0533
Tau=0.3 | Sample=02 | Repeat=3/3 | Latency=0.2394s | RTF=0.0537
✅ Sample 02 completed | Mean latency=0.2548s | RTF=0.0571

Tau=0.3 | Sample=03 | Repeat=1/3 | Latency=0.6384s | RTF=0.1010
Tau=0.3 | Sample=03 | Repeat=2/3 | Latency=0.2945s | RTF=0.0466
Tau=0.3 | Sample=03 | Repeat=3/3 | Latency=0.2884s | RTF=0.0456
✅ Sample 03 completed | Mean latency=0.4071s | RTF=0.0644

Tau=0.3 | Sample=04 | Repeat=1/3 | Latency=0.2429s | RTF=0.0573
Tau=0.3 | Sample=04 | Repeat=2/3 | Latency

In [ ]:
# ============================================================
# FINAL COMPARISON — STEP 7: INSTALL SEED-VC
# ============================================================

import os
import sys
import json
import subprocess

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

COMPARISON_DIR = os.path.join(
    BASE_DIR,
    "final_hybrid_comparison"
)

os.makedirs(COMPARISON_DIR, exist_ok=True)

# حفظ معلومات OpenVoice المستخدمة
environment_metadata = {
    "openvoice_repository": "myshell-ai/OpenVoice",
    "openvoice_commit": "74a1d147b17a8c3092dd5430504bd83ef6c7eb23",
    "openvoice_checkpoint": (
        "/content/drive/MyDrive/IJEEE_VC_data/"
        "models/OpenVoiceV2/converter/checkpoint.pth"
    ),
    "seed_vc_repository": "Plachtaa/seed-vc",
    "seed_vc_commit": "51383efd921027683c89e5348211d93ff12ac2a8",
    "gpu": "NVIDIA Tesla T4",
    "holdout_samples": 10
}

with open(
    os.path.join(
        COMPARISON_DIR,
        "experiment_environment.json"
    ),
    "w",
    encoding="utf-8"
) as metadata_file:

    json.dump(
        environment_metadata,
        metadata_file,
        indent=2
    )

# منع Transformers من استيراد TensorFlow
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

SEED_REPO = "/content/seed-vc"
SEED_COMMIT = "51383efd921027683c89e5348211d93ff12ac2a8"

if not os.path.isdir(SEED_REPO):

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Plachtaa/seed-vc.git",
            SEED_REPO
        ],
        check=True
    )

subprocess.run(
    ["git", "checkout", SEED_COMMIT],
    cwd=SEED_REPO,
    check=True
)

seed_packages = [
    "numpy==1.26.4",
    "scipy==1.13.1",
    "librosa==0.10.2",
    "transformers==4.46.3",
    "huggingface-hub>=0.28.1",
    "munch==4.0.0",
    "einops==0.8.0",
    "descript-audio-codec==1.0.0",
    "resemblyzer",
    "soundfile==0.12.1",
    "modelscope==1.18.1",
    "funasr==1.1.5",
    "hydra-core==1.3.2",
    "python-dotenv",
    "pyyaml"
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q"
    ] + seed_packages,
    check=True
)

installed_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"
    cwd=SEED_REPO,
    text=True
).strip()

print("=" * 70)
print("SEED-VC INSTALLATION COMPLETE")
print("=" * 70)

print("Repository:", SEED_REPO)
print("Commit:", installed_commit)
print("Environment metadata saved")
print("\n✅ Seed-VC installed")
print("⚠️ الآن اعملي Runtime → Restart session")

SEED-VC INSTALLATION COMPLETE
Repository: /content/seed-vc
Commit: 51383efd921027683c89e5348211d93ff12ac2a8
Environment metadata saved

✅ Seed-VC installed
⚠️ الآن اعملي Runtime → Restart session


In [ ]:
# ============================================================
# RECOVERY — STEP 1: CHECK SAVED COMPARISON DATA
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import glob
import torch

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

HOLDOUT_DIR = os.path.join(
    BASE_DIR,
    "final_holdout_data"
)

COMPARISON_DIR = os.path.join(
    BASE_DIR,
    "final_hybrid_comparison"
)

TAU_03_DIR = os.path.join(
    COMPARISON_DIR,
    "openvoice_outputs",
    "tau_0.3"
)

TAU_05_DIR = os.path.join(
    COMPARISON_DIR,
    "openvoice_outputs",
    "tau_0.5"
)

HOLDOUT_FILES = sorted(
    glob.glob(
        os.path.join(
            HOLDOUT_DIR,
            "joe_holdout_*.wav"
        )
    )
)

TAU_03_FILES = sorted(
    glob.glob(
        os.path.join(
            TAU_03_DIR,
            "openvoice_sample_*_tau_0.3.wav"
        )
    )
)

TAU_05_FILES = sorted(
    glob.glob(
        os.path.join(
            TAU_05_DIR,
            "openvoice_sample_*_tau_0.5.wav"
        )
    )
)

METADATA_PATH = os.path.join(
    HOLDOUT_DIR,
    "holdout_metadata.csv"
)

print("=" * 70)
print("FINAL COMPARISON RECOVERY CHECK")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nOriginal holdout samples:", len(HOLDOUT_FILES))
print("OpenVoice tau=0.3 outputs:", len(TAU_03_FILES))
print("OpenVoice tau=0.5 outputs:", len(TAU_05_FILES))
print("Holdout metadata exists:", os.path.isfile(METADATA_PATH))

assert torch.cuda.is_available(), (
    "❌ اختاري T4 GPU"
)

assert len(HOLDOUT_FILES) == 10, (
    "❌ Expected 10 original holdout samples"
)

assert len(TAU_03_FILES) == 10, (
    "❌ Expected 10 tau=0.3 outputs"
)

assert len(TAU_05_FILES) == 10, (
    "❌ Expected 10 tau=0.5 outputs"
)

assert os.path.isfile(METADATA_PATH), (
    "❌ Holdout metadata is missing"
)

print("\n✅ All previous work is safely stored on Drive")
print("Next: reinstall Seed-VC only")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FINAL COMPARISON RECOVERY CHECK
CUDA available: True
GPU: Tesla T4

Original holdout samples: 10
OpenVoice tau=0.3 outputs: 10
OpenVoice tau=0.5 outputs: 10
Holdout metadata exists: True

✅ All previous work is safely stored on Drive
Next: reinstall Seed-VC only


In [ ]:
# ============================================================
# RECOVERY — STEP 2: REINSTALL SEED-VC ONLY
# ============================================================

import os
import sys
import subprocess

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

SEED_REPO = "/content/seed-vc"
SEED_COMMIT = "51383efd921027683c89e5348211d93ff12ac2a8"

# Clone the fixed repository
if not os.path.isdir(SEED_REPO):

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Plachtaa/seed-vc.git",
            SEED_REPO
        ],
        check=True
    )

# Use the same fixed commit
subprocess.run(
    ["git", "checkout", SEED_COMMIT],
    cwd=SEED_REPO,
    check=True
)

# Install Seed dependencies without replacing Colab Torch
seed_packages = [
    "numpy==1.26.4",
    "scipy==1.13.1",
    "librosa==0.10.2",
    "transformers==4.46.3",
    "huggingface-hub>=0.28.1",
    "munch==4.0.0",
    "einops==0.8.0",
    "descript-audio-codec==1.0.0",
    "resemblyzer",
    "soundfile==0.12.1",
    "modelscope==1.18.1",
    "funasr==1.1.5",
    "hydra-core==1.3.2",
    "python-dotenv",
    "pyyaml"
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q"
    ] + seed_packages,
    check=True
)

installed_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=SEED_REPO,
    text=True
).strip()

print("=" * 70)
print("SEED-VC RECOVERY INSTALLATION COMPLETE")
print("=" * 70)

print("Repository:", SEED_REPO)
print("Commit:", installed_commit)
print("Python:", sys.version.split()[0])

print("\n✅ Seed-VC installed successfully")
print("⚠️ الآن اعملي Runtime → Restart session")

SEED-VC RECOVERY INSTALLATION COMPLETE
Repository: /content/seed-vc
Commit: 51383efd921027683c89e5348211d93ff12ac2a8
Python: 3.13.15

✅ Seed-VC installed successfully
⚠️ الآن اعملي Runtime → Restart session


In [ ]:
# ============================================================
# RECOVERY — STEP 3: POST-RESTART SEED CHECK
# ============================================================

import os

# يجب وضعها قبل استيراد Transformers
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

from google.colab import drive
drive.mount("/content/drive")

import sys
import glob
import torch
import numpy as np
import librosa
import transformers

SEED_REPO = "/content/seed-vc"

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

HOLDOUT_DIR = os.path.join(
    BASE_DIR,
    "final_holdout_data"
)

COMPARISON_DIR = os.path.join(
    BASE_DIR,
    "final_hybrid_comparison"
)

TAU_03_DIR = os.path.join(
    COMPARISON_DIR,
    "openvoice_outputs",
    "tau_0.3"
)

TAU_05_DIR = os.path.join(
    COMPARISON_DIR,
    "openvoice_outputs",
    "tau_0.5"
)

HOLDOUT_FILES = sorted(
    glob.glob(
        os.path.join(
            HOLDOUT_DIR,
            "joe_holdout_*.wav"
        )
    )
)

TAU_03_FILES = sorted(
    glob.glob(
        os.path.join(
            TAU_03_DIR,
            "openvoice_sample_*_tau_0.3.wav"
        )
    )
)

TAU_05_FILES = sorted(
    glob.glob(
        os.path.join(
            TAU_05_DIR,
            "openvoice_sample_*_tau_0.5.wav"
        )
    )
)

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

if SEED_REPO not in sys.path:
    sys.path.insert(0, SEED_REPO)

print("=" * 70)
print("POST-RESTART SEED ENVIRONMENT CHECK")
print("=" * 70)

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("NumPy:", np.__version__)
print("Librosa:", librosa.__version__)
print("Transformers:", transformers.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("TensorFlow enabled:", transformers.utils.is_tf_available())
print("Seed repository exists:", os.path.isdir(SEED_REPO))
print("Kathleen reference exists:", os.path.isfile(TARGET_PATH))

print("\nOriginal holdout samples:", len(HOLDOUT_FILES))
print("OpenVoice tau=0.3 files:", len(TAU_03_FILES))
print("OpenVoice tau=0.5 files:", len(TAU_05_FILES))

assert torch.cuda.is_available()
assert os.path.isdir(SEED_REPO)
assert os.path.isfile(TARGET_PATH)
assert len(HOLDOUT_FILES) == 10
assert len(TAU_03_FILES) == 10
assert len(TAU_05_FILES) == 10
assert transformers.__version__ == "4.46.3"

print("\n✅ Seed environment and all comparison inputs are ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
POST-RESTART SEED ENVIRONMENT CHECK
Python: 3.13.15
Torch: 2.11.0+cu128
NumPy: 1.26.4
Librosa: 0.10.2
Transformers: 4.46.3

CUDA available: True
GPU: Tesla T4
TensorFlow enabled: False
Seed repository exists: True
Kathleen reference exists: True

Original holdout samples: 10
OpenVoice tau=0.3 files: 10
OpenVoice tau=0.5 files: 10

✅ Seed environment and all comparison inputs are ready


In [ ]:
# ============================================================
# FINAL COMPARISON — STEP 8: LOAD SEED-VC MODELS
# ============================================================

import os
import sys
import torch

from types import SimpleNamespace

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

SEED_REPO = "/content/seed-vc"

if SEED_REPO not in sys.path:
    sys.path.insert(0, SEED_REPO)

# Seed-VC uses relative repository paths
os.chdir(SEED_REPO)

import inference as seed_inference

seed_model_args = SimpleNamespace(
    source=None,
    target=None,
    output=None,

    diffusion_steps=10,
    length_adjust=1.0,
    inference_cfg_rate=0.7,

    f0_condition=False,
    auto_f0_adjust=False,
    semi_tone_shift=0,

    checkpoint=None,
    config=None,
    fp16=True
)

print("Loading Seed-VC models...")
print(
    "The first load may download Seed, Whisper, "
    "CAM++ and BigVGAN.\n"
)

torch.cuda.empty_cache()

seed_model_bundle = seed_inference.load_models(
    seed_model_args
)

print("\n" + "=" * 70)
print("SEED-VC MODELS READY")
print("=" * 70)

print("✅ Seed-VC models loaded")
print("Loaded components:", len(seed_model_bundle))
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Model bundle:", "seed_model_bundle")

Loading Seed-VC models...
The first load may download Seed, Whisper, CAM++ and BigVGAN.



DiT_seed_v2_uvit_whisper_small_wavenet_b(…):   0%|          | 0.00/440M [00:00<?, ?B/s]

(…)_mel_seed_uvit_whisper_small_wavenet.yml: 0.00B [00:00, ?B/s]

cfm loaded
length_regulator loaded


campplus_cn_common.bin:   0%|          | 0.00/28.0M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]


SEED-VC MODELS READY
✅ Seed-VC models loaded
Loaded components: 7
CUDA available: True
GPU: Tesla T4
Model bundle: seed_model_bundle


In [ ]:
# ============================================================
# FINAL COMPARISON — STEP 9: SEED-VC WARM-UP
# ============================================================

import os
import random
import numpy as np
import torch

from types import SimpleNamespace

WARMUP_DIR = "/content/final_seed_warmup"
os.makedirs(WARMUP_DIR, exist_ok=True)

assert "seed_model_bundle" in globals()
assert len(HOLDOUT_FILES) == 10
assert os.path.isfile(TARGET_PATH)

# Prevent Seed main() from loading models again
if "original_seed_load_models" not in globals():
    original_seed_load_models = (
        seed_inference.load_models
    )

seed_inference.load_models = (
    lambda args: seed_model_bundle
)

warmup_args = SimpleNamespace(
    source=HOLDOUT_FILES[0],
    target=TARGET_PATH,
    output=WARMUP_DIR,

    diffusion_steps=10,
    length_adjust=1.0,
    inference_cfg_rate=0.7,

    f0_condition=False,
    auto_f0_adjust=False,
    semi_tone_shift=0,

    checkpoint=None,
    config=None,
    fp16=True
)

print("Running Seed-VC warm-up...\n")

for warmup_run in range(1, 3):

    random.seed(2028)
    np.random.seed(2028)
    torch.manual_seed(2028)
    torch.cuda.manual_seed_all(2028)

    torch.cuda.synchronize()

    print(f"Warm-up run {warmup_run}/2")

    seed_inference.main(
        warmup_args
    )

    torch.cuda.synchronize()

    print(
        f"✅ Warm-up run {warmup_run} completed\n"
    )

print("=" * 70)
print("✅ SEED-VC WARM-UP COMPLETED")
print("=" * 70)

print("Warm-up steps: 10")
print("CFG rate: 0.7")
print("GPU:", torch.cuda.get_device_name(0))

Running Seed-VC warm-up...

Warm-up run 1/2


It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
100%|██████████| 10/10 [00:00<00:00, 10.22it/s]


RTF: 0.8578664206499321


It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


✅ Warm-up run 1 completed

Warm-up run 2/2


100%|██████████| 10/10 [00:00<00:00, 20.69it/s]


RTF: 0.23405684776186386
✅ Warm-up run 2 completed

✅ SEED-VC WARM-UP COMPLETED
Warm-up steps: 10
CFG rate: 0.7
GPU: Tesla T4


In [ ]:
# ============================================================
# FINAL COMPARISON — STEP 10:
# RUN SEED-ONLY, DEFAULT HYBRID AND OPTIMIZED HYBRID
# ============================================================

import os
import glob
import time
import shutil
import random

import numpy as np
import pandas as pd
import torch

from types import SimpleNamespace

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

HOLDOUT_DIR = os.path.join(
    BASE_DIR,
    "final_holdout_data"
)

COMPARISON_DIR = os.path.join(
    BASE_DIR,
    "final_hybrid_comparison"
)

OPENVOICE_OUTPUT_DIR = os.path.join(
    COMPARISON_DIR,
    "openvoice_outputs"
)

SEED_OUTPUT_DIR = os.path.join(
    COMPARISON_DIR,
    "seed_based_outputs"
)

LOCAL_WORK_DIR = "/content/final_seed_comparison_work"

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

METADATA_PATH = os.path.join(
    HOLDOUT_DIR,
    "holdout_metadata.csv"
)

OPENVOICE_SAMPLE_METRICS_PATH = os.path.join(
    COMPARISON_DIR,
    "openvoice_final_sample_metrics.csv"
)

RAW_SEED_METRICS_PATH = os.path.join(
    COMPARISON_DIR,
    "seed_final_timing_repetitions.csv"
)

SEED_SAMPLE_METRICS_PATH = os.path.join(
    COMPARISON_DIR,
    "seed_final_sample_metrics.csv"
)

FOUR_SYSTEM_METRICS_PATH = os.path.join(
    COMPARISON_DIR,
    "four_system_timing_sample_metrics.csv"
)

FOUR_SYSTEM_SUMMARY_PATH = os.path.join(
    COMPARISON_DIR,
    "four_system_timing_summary.csv"
)

os.makedirs(SEED_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

# ------------------------------------------------------------
# Input files
# ------------------------------------------------------------

HOLDOUT_FILES = sorted(
    glob.glob(
        os.path.join(
            HOLDOUT_DIR,
            "joe_holdout_*.wav"
        )
    )
)

TAU_03_FILES = sorted(
    glob.glob(
        os.path.join(
            OPENVOICE_OUTPUT_DIR,
            "tau_0.3",
            "openvoice_sample_*_tau_0.3.wav"
        )
    )
)

TAU_05_FILES = sorted(
    glob.glob(
        os.path.join(
            OPENVOICE_OUTPUT_DIR,
            "tau_0.5",
            "openvoice_sample_*_tau_0.5.wav"
        )
    )
)

assert len(HOLDOUT_FILES) == 10
assert len(TAU_03_FILES) == 10
assert len(TAU_05_FILES) == 10
assert os.path.isfile(TARGET_PATH)
assert os.path.isfile(METADATA_PATH)
assert os.path.isfile(OPENVOICE_SAMPLE_METRICS_PATH)
assert "seed_model_bundle" in globals()
assert torch.cuda.is_available()

# Prevent model reloading
seed_inference.load_models = (
    lambda args: seed_model_bundle
)

# ------------------------------------------------------------
# Metadata and OpenVoice latency
# ------------------------------------------------------------

holdout_df = pd.read_csv(
    METADATA_PATH
)

openvoice_metrics_df = pd.read_csv(
    OPENVOICE_SAMPLE_METRICS_PATH
)

duration_map = {
    int(row["sample"]): float(row["duration_seconds"])
    for _, row in holdout_df.iterrows()
}

openvoice_latency_map = {
    (
        round(float(row["tau"]), 1),
        int(row["sample"])
    ): float(row["mean_openvoice_latency_seconds"])
    for _, row in openvoice_metrics_df.iterrows()
}

# ------------------------------------------------------------
# Systems under evaluation
# ------------------------------------------------------------

SYSTEMS = [
    {
        "system": "seed_only",
        "display_name": "Seed-VC Only",
        "sources": HOLDOUT_FILES,
        "tau": None,
        "steps": 10
    },
    {
        "system": "default_hybrid",
        "display_name": "Default Hybrid",
        "sources": TAU_03_FILES,
        "tau": 0.3,
        "steps": 20
    },
    {
        "system": "optimized_hybrid",
        "display_name": "Optimized Hybrid",
        "sources": TAU_05_FILES,
        "tau": 0.5,
        "steps": 10
    }
]

TIMING_REPEATS = 3

SAMPLE_SEEDS = {
    sample: 202900 + sample
    for sample in range(1, 11)
}

# ------------------------------------------------------------
# Resume support
# ------------------------------------------------------------

if os.path.isfile(RAW_SEED_METRICS_PATH):
    existing_raw_df = pd.read_csv(
        RAW_SEED_METRICS_PATH
    )
    raw_records = existing_raw_df.to_dict("records")
else:
    raw_records = []

if os.path.isfile(SEED_SAMPLE_METRICS_PATH):
    existing_sample_df = pd.read_csv(
        SEED_SAMPLE_METRICS_PATH
    )
    sample_records = existing_sample_df.to_dict("records")

    completed_cases = {
        (
            str(row["system"]),
            int(row["sample"])
        )
        for _, row in existing_sample_df.iterrows()
    }
else:
    sample_records = []
    completed_cases = set()

print("=" * 80)
print("FINAL FOUR-SYSTEM HOLDOUT COMPARISON")
print("=" * 80)

print("Holdout samples:", 10)
print("Seed timing repetitions:", TIMING_REPEATS)
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Run Seed-based systems
# ------------------------------------------------------------

for system_config in SYSTEMS:

    system_name = system_config["system"]
    display_name = system_config["display_name"]
    input_sources = system_config["sources"]
    tau = system_config["tau"]
    steps = system_config["steps"]

    system_drive_dir = os.path.join(
        SEED_OUTPUT_DIR,
        system_name
    )

    system_local_dir = os.path.join(
        LOCAL_WORK_DIR,
        system_name
    )

    os.makedirs(system_drive_dir, exist_ok=True)
    os.makedirs(system_local_dir, exist_ok=True)

    print("\n" + "=" * 80)
    print(display_name)
    print(f"Tau: {tau} | Diffusion steps: {steps}")
    print("=" * 80)

    for sample_number in range(1, 11):

        case_key = (
            system_name,
            sample_number
        )

        if case_key in completed_cases:
            print(
                f"Skipping completed case: "
                f"{system_name}, sample {sample_number:02d}"
            )
            continue

        # Remove incomplete repetitions from an interrupted run
        raw_records = [
            record
            for record in raw_records
            if not (
                str(record["system"]) == system_name
                and int(record["sample"]) == sample_number
            )
        ]

        source_path = input_sources[
            sample_number - 1
        ]

        duration = duration_map[
            sample_number
        ]

        fixed_seed = SAMPLE_SEEDS[
            sample_number
        ]

        local_sample_dir = os.path.join(
            system_local_dir,
            f"sample_{sample_number:02d}"
        )

        os.makedirs(
            local_sample_dir,
            exist_ok=True
        )

        run_args = SimpleNamespace(
            source=source_path,
            target=TARGET_PATH,
            output=local_sample_dir,

            diffusion_steps=steps,
            length_adjust=1.0,
            inference_cfg_rate=0.7,

            f0_condition=False,
            auto_f0_adjust=False,
            semi_tone_shift=0,

            checkpoint=None,
            config=None,
            fp16=True
        )

        repetition_latencies = []

        for repetition in range(
            1,
            TIMING_REPEATS + 1
        ):

            random.seed(fixed_seed)
            np.random.seed(fixed_seed)
            torch.manual_seed(fixed_seed)
            torch.cuda.manual_seed_all(fixed_seed)

            torch.cuda.synchronize()
            start_time = time.perf_counter()

            seed_inference.main(
                run_args
            )

            torch.cuda.synchronize()
            seed_latency = (
                time.perf_counter() - start_time
            )

            seed_rtf = (
                seed_latency / duration
            )

            repetition_latencies.append(
                seed_latency
            )

            raw_records.append({
                "system": system_name,
                "tau": tau,
                "diffusion_steps": steps,
                "sample": sample_number,
                "repetition": repetition,
                "fixed_seed": fixed_seed,
                "duration_seconds": duration,
                "seed_latency_seconds": seed_latency,
                "seed_rtf": seed_rtf
            })

            pd.DataFrame(raw_records).to_csv(
                RAW_SEED_METRICS_PATH,
                index=False
            )

            print(
                f"{display_name} | "
                f"Sample={sample_number:02d} | "
                f"Repeat={repetition}/{TIMING_REPEATS} | "
                f"Seed latency={seed_latency:.4f}s | "
                f"RTF={seed_rtf:.4f}"
            )

        mean_seed_latency = float(
            np.mean(repetition_latencies)
        )

        std_seed_latency = float(
            np.std(
                repetition_latencies,
                ddof=1
            )
        )

        if tau is None:
            openvoice_latency = 0.0
        else:
            openvoice_latency = (
                openvoice_latency_map[
                    (
                        round(float(tau), 1),
                        sample_number
                    )
                ]
            )

        total_latency = (
            openvoice_latency
            + mean_seed_latency
        )

        total_rtf = (
            total_latency / duration
        )

        generated_files = sorted(
            glob.glob(
                os.path.join(
                    local_sample_dir,
                    "*.wav"
                )
            ),
            key=os.path.getmtime
        )

        assert generated_files, (
            f"❌ No Seed output for "
            f"{system_name}, sample {sample_number}"
        )

        if system_name == "seed_only":

            final_filename = (
                f"seed_only_sample_{sample_number:02d}_"
                f"steps_10.wav"
            )

        elif system_name == "default_hybrid":

            final_filename = (
                f"default_hybrid_sample_{sample_number:02d}_"
                f"tau_0.3_steps_20.wav"
            )

        else:

            final_filename = (
                f"optimized_hybrid_sample_{sample_number:02d}_"
                f"tau_0.5_steps_10.wav"
            )

        final_output_path = os.path.join(
            system_drive_dir,
            final_filename
        )

        shutil.copy2(
            generated_files[-1],
            final_output_path
        )

        sample_records.append({
            "system": system_name,
            "display_name": display_name,
            "tau": tau,
            "diffusion_steps": steps,
            "sample": sample_number,
            "fixed_seed": fixed_seed,
            "duration_seconds": duration,
            "openvoice_latency_seconds": openvoice_latency,
            "mean_seed_latency_seconds": mean_seed_latency,
            "std_seed_latency_seconds": std_seed_latency,
            "total_latency_seconds": total_latency,
            "total_rtf": total_rtf,
            "output_path": final_output_path
        })

        pd.DataFrame(sample_records).to_csv(
            SEED_SAMPLE_METRICS_PATH,
            index=False
        )

        completed_cases.add(case_key)

        print(
            f"\n✅ {display_name}, sample {sample_number:02d} completed"
        )
        print(
            f"Mean Seed latency: {mean_seed_latency:.4f}s"
        )
        print(
            f"Total latency: {total_latency:.4f}s"
        )
        print(
            f"Total RTF: {total_rtf:.4f}\n"
        )

# ------------------------------------------------------------
# Add OpenVoice-only timing rows
# ------------------------------------------------------------

seed_sample_df = pd.DataFrame(
    sample_records
)

openvoice_only_df = (
    openvoice_metrics_df[
        openvoice_metrics_df["tau"].round(1) == 0.5
    ]
    .copy()
)

openvoice_only_rows = pd.DataFrame({
    "system": "openvoice_only",
    "display_name": "OpenVoice Only",
    "tau": 0.5,
    "diffusion_steps": np.nan,
    "sample": openvoice_only_df["sample"].astype(int),
    "fixed_seed": openvoice_only_df["fixed_seed"],
    "duration_seconds": openvoice_only_df["duration_seconds"],
    "openvoice_latency_seconds": (
        openvoice_only_df[
            "mean_openvoice_latency_seconds"
        ]
    ),
    "mean_seed_latency_seconds": 0.0,
    "std_seed_latency_seconds": 0.0,
    "total_latency_seconds": (
        openvoice_only_df[
            "mean_openvoice_latency_seconds"
        ]
    ),
    "total_rtf": (
        openvoice_only_df[
            "mean_openvoice_rtf"
        ]
    ),
    "output_path": (
        openvoice_only_df["output_path"]
    )
})

four_system_df = pd.concat(
    [
        openvoice_only_rows,
        seed_sample_df
    ],
    ignore_index=True
)

system_order = [
    "openvoice_only",
    "seed_only",
    "default_hybrid",
    "optimized_hybrid"
]

four_system_df["system"] = pd.Categorical(
    four_system_df["system"],
    categories=system_order,
    ordered=True
)

four_system_df = four_system_df.sort_values(
    ["system", "sample"]
)

four_system_df.to_csv(
    FOUR_SYSTEM_METRICS_PATH,
    index=False
)

# ------------------------------------------------------------
# Final timing summary
# ------------------------------------------------------------

timing_summary_df = (
    four_system_df
    .groupby(
        ["system", "display_name"],
        observed=True,
        as_index=False
    )
    .agg(
        mean_latency=(
            "total_latency_seconds",
            "mean"
        ),
        std_latency=(
            "total_latency_seconds",
            "std"
        ),
        mean_rtf=(
            "total_rtf",
            "mean"
        ),
        std_rtf=(
            "total_rtf",
            "std"
        )
    )
)

timing_summary_df.to_csv(
    FOUR_SYSTEM_SUMMARY_PATH,
    index=False
)

print("\n" + "=" * 80)
print("FINAL FOUR-SYSTEM TIMING SUMMARY")
print("=" * 80)

print(
    timing_summary_df.round(4).to_string(
        index=False
    )
)

print("\n✅ Seed outputs:", SEED_OUTPUT_DIR)
print("✅ Sample metrics:", FOUR_SYSTEM_METRICS_PATH)
print("✅ Timing summary:", FOUR_SYSTEM_SUMMARY_PATH)

It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


FINAL FOUR-SYSTEM HOLDOUT COMPARISON
Holdout samples: 10
Seed timing repetitions: 3
GPU: Tesla T4

Seed-VC Only
Tau: None | Diffusion steps: 10


100%|██████████| 10/10 [00:00<00:00, 19.20it/s]
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


RTF: 0.2706396435909669
Seed-VC Only | Sample=01 | Repeat=1/3 | Seed latency=1.3733s | RTF=0.2758


100%|██████████| 10/10 [00:00<00:00, 20.67it/s]
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


RTF: 0.22954811825129753
Seed-VC Only | Sample=01 | Repeat=2/3 | Seed latency=1.1511s | RTF=0.2311


... (172 outputs left out)

100%|██████████| 10/10 [00:00<00:00, 17.35it/s]
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


RTF: 0.22828256478533146
Optimized Hybrid | Sample=10 | Repeat=2/3 | Seed latency=1.5705s | RTF=0.2296


100%|██████████| 10/10 [00:00<00:00, 17.36it/s]


RTF: 0.22769614588469267
Optimized Hybrid | Sample=10 | Repeat=3/3 | Seed latency=1.5657s | RTF=0.2289

✅ Optimized Hybrid, sample 10 completed
Mean Seed latency: 1.6372s
Total latency: 1.9376s
Total RTF: 0.2833


FINAL FOUR-SYSTEM TIMING SUMMARY
          system     display_name  mean_latency  std_latency  mean_rtf  std_rtf
  openvoice_only   OpenVoice Only        0.2622       0.0259    0.0501   0.0047
       seed_only     Seed-VC Only        1.3749       0.1995    0.2609   0.0202
  default_hybrid   Default Hybrid        2.2105       0.2986    0.4196   0.0243
optimized_hybrid Optimized Hybrid        1.6511       0.2043    0.3139   0.0202

✅ Seed outputs: /content/drive/MyDrive/IJEEE_VC_data/final_hybrid_comparison/seed_based_outputs
✅ Sample metrics: /content/drive/MyDrive/IJEEE_VC_data/final_hybrid_comparison/four_system_timing_sample_metrics.csv
✅ Timing summary: /content/drive/MyDrive/IJEEE_VC_data/final_hybrid_comparison/four_system_timing_summary.csv
